In [ ]:
%pip install opentelemetry-exporter-otlp

In [ ]:
%pip install opentelemetry-api
%pip install opentelemetry-sdk


In [1]:
from opentelemetry.sdk.metrics import MeterProvider
from opentelemetry.sdk.metrics.export import (
    PeriodicExportingMetricReader,
)
from opentelemetry.exporter.otlp.proto.grpc.metric_exporter import OTLPMetricExporter
from opentelemetry.sdk.resources import SERVICE_NAME, Resource
from opentelemetry.metrics import set_meter_provider, get_meter_provider

# 1. Configure resource
resource = Resource.create(attributes={SERVICE_NAME: "TestPythonService"})

# 2. Set up the exporter and reader
metric_reader = PeriodicExportingMetricReader(
    OTLPMetricExporter(
        endpoint="http://localhost:4317/v1/metrics",
        insecure=True
    ),
    export_interval_millis=500
)

# 3. Create and set MeterProvider BEFORE calling get_meter_provider or get_meter
provider = MeterProvider(resource=resource, metric_readers=[metric_reader])
set_meter_provider(provider)

# 4. Now it's safe to access meters
meter = get_meter_provider().get_meter("TestPythonService")


In [ ]:
import time

# 4. Now it's safe to access meters

counter = meter.create_counter("example_counter")
counter.add(1, {"region": "us-east"})

while True:
    counter.add(1, {"region": "us-east"})
    time.sleep(0.5)

In [2]:
import threading, time
#from opentelemetry.sdk.metrics import Meter 

gauge = meter.create_gauge("ServiceGauge")

value = 50
direction = 1

def update_gauge():
    while True:
        
        global value
        global direction
        
        value += direction
        if value >= 100:
            direction = -1
        elif value <= 50:
            direction = 1

        gauge.set(value, {"service": "TestPythonService"})

        time.sleep(1)

threading.Thread(target=update_gauge, daemon=True).start()
